## 99acres Gandhinagar Property Scraper
Scrapes all property listings from 99acres, parses the HTML, and saves a clean structured CSV.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import random


options = Options()

options.add_argument("--start-maximized")

# Helps reduce bot detection
options.add_argument("--disable-blink-features=AutomationControlled")

options.add_argument(
    "user-agent=Mozilla/5.0 "
    "(Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 "
    "(KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=options)

all_data = []

TOTAL_PAGES = 71

for page in range(1, TOTAL_PAGES + 1):

    if page == 1:
        url = "https://www.99acres.com/property-in-gandhinagar-ffid"
    else:
        url = f"https://www.99acres.com/property-in-gandhinagar-ffid-page-{page}"

    print(f"\nOpening Page {page}/{TOTAL_PAGES}")

    driver.get(url)

    try:

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located(
                (By.CLASS_NAME, "tupleNew__contentWrap")
            )
        )

    except:

        print(f"Failed loading page {page}")
        continue

    last_count = 0

    while True:

        # Get current page height
        page_height = driver.execute_script(
            "return document.body.scrollHeight"
        )

        current_position = 0

        # Smooth fast scrolling
        while current_position < page_height:

            driver.execute_script(
                f"window.scrollTo(0, {current_position});"
            )

            current_position += 600

            time.sleep(0.7)

        # Wait once for lazy loading
        time.sleep(4)

        # Count cards
        cards = driver.find_elements(
            By.CLASS_NAME,
            "tupleNew__contentWrap"
        )

        current_count = len(cards)

        print(f"Loaded Properties: {current_count}")

        # If no new cards loaded -> stop immediately
        if current_count == last_count:

            print("All properties loaded")
            break

        # Update count
        last_count = current_count


    soup = BeautifulSoup(driver.page_source, "html.parser")

    property_cards = soup.find_all(
        "div",
        class_="tupleNew__contentWrap"
    )

    print(f"Final Property Count: {len(property_cards)}")

    # get data from each card
    for card in property_cards:

        try:
            
            name = ""

            name_tag = card.find("a")

            if name_tag:
                name = name_tag.get_text(strip=True)


            property_url = ""

            if name_tag and name_tag.has_attr("href"):

                property_url = name_tag["href"]

                if property_url.startswith("/"):
                    property_url = (
                        "https://www.99acres.com"
                        + property_url
                    )

            image_url = ""

            img_tag = card.find("img")

            if img_tag:

                if img_tag.get("src"):
                    image_url = img_tag["src"]

                elif img_tag.get("data-src"):
                    image_url = img_tag["data-src"]


            raw_text = card.get_text(
                separator=" | ",
                strip=True
            )

            parts = [
                p.strip()
                for p in raw_text.split("|")
            ]

            location = ""
            bhk = ""
            price = ""
            price_per_sqft = ""
            area_sqft = ""
            area_sqm = ""
            area_type = ""
            status = ""
            description = ""
            builder = ""
            posted_time = ""
            
            
            for p in parts:

                # BHK
                if "BHK" in p and not bhk:
                    bhk = p

                # LOCATION
                elif "in " in p and not location:
                    location = p.replace("in ", "").strip()

                # PRICE
                elif "₹" in p and "/sqft" not in p:
                    price = p

                # PRICE PER SQFT
                elif "/sqft" in p:
                    price_per_sqft = p

                # AREA SQFT
                elif "sqft" in p and "(" not in p:
                    area_sqft = p

                # AREA SQM
                elif "sqm" in p:
                    area_sqm = p

                # AREA TYPE
                elif (
                    "Built-up" in p
                    or "Carpet" in p
                    or "Super Built-up" in p
                ):
                    area_type = p

                # STATUS
                elif (
                    "Construction" in p
                    or "Ready To Move" in p
                ):
                    status = p

                # BUILDER
                elif "Builder" in p:
                    builder = p

                # POSTED TIME
                elif "ago" in p:
                    posted_time = p

                # DESCRIPTION
                elif len(p) > 60:
                    description = p

            all_data.append({

                "Page": page,
                "Name": name,
                "BHK": bhk,
                "Location": location,
                "Price": price,
                "Price_per_sqft": price_per_sqft,
                "Area_sqft": area_sqft,
                "Area_sqm": area_sqm,
                "Area_type": area_type,
                "Status": status,
                "Description": description,
                "Builder": builder,
                "Posted_Time": posted_time,
                "Property_URL": property_url,
                "Image_URL": image_url

            })

        except Exception as e:

            print("Card Error:", e)

    # Delay between pages
    time.sleep(random.uniform(3, 6))


driver.quit()

df = pd.DataFrame(all_data)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Reset index
df.reset_index(drop=True, inplace=True)


df.to_csv(
    "gandhinagar_properties_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n====================================")
print("SCRAPING COMPLETED")
print("====================================")

print(f"Total Properties Scraped: {len(df)}")

print("\nCSV Saved Successfully")


Opening Page 1/52
Loaded Properties: 2
Loaded Properties: 72
Loaded Properties: 72
All properties loaded
Final Property Count: 72

Opening Page 2/52
Loaded Properties: 12
Loaded Properties: 70
Loaded Properties: 70
All properties loaded
Final Property Count: 70

Opening Page 3/52
Loaded Properties: 9
Loaded Properties: 58
Loaded Properties: 58
All properties loaded
Final Property Count: 58

Opening Page 4/52
Loaded Properties: 24
Loaded Properties: 49
Loaded Properties: 49
All properties loaded
Final Property Count: 49

Opening Page 5/52
Loaded Properties: 25
Loaded Properties: 25
All properties loaded
Final Property Count: 25

Opening Page 6/52
Loaded Properties: 25
Loaded Properties: 25
All properties loaded
Final Property Count: 25

Opening Page 7/52
Loaded Properties: 25
Loaded Properties: 25
All properties loaded
Final Property Count: 25

Opening Page 8/52
Loaded Properties: 25
Loaded Properties: 25
All properties loaded
Final Property Count: 25

Opening Page 9/52
Loaded Properti